# Análisis de Resultados de Simulación

Este notebook carga y analiza los resultados de los experimentos de simulación.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


## Cargar Resultados


In [ ]:
# Cargar todos los archivos CSV de resultados
results_dir = "../results"

all_data = []
for file in os.listdir(results_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(results_dir, file))
        all_data.append(df)

if all_data:
    df_all = pd.concat(all_data, ignore_index=True)
    print(f"Total de réplicas cargadas: {len(df_all)}")
    print(f"\nColumnas: {list(df_all.columns)}")
    print(f"\nPrimeras filas:")
    display(df_all.head())
else:
    print("No se encontraron archivos de resultados.")


## Análisis por Estrategia


In [ ]:
if 'df_all' in locals():
    # Agrupar por estrategia
    strategy_summary = df_all.groupby('strategy').agg({
        'mean_search_time': ['mean', 'std'],
        'p95_search_time': ['mean', 'std'],
        'error_rate': ['mean', 'std'],
        'avg_utilization': ['mean', 'std'],
        'orders_processed': 'sum'
    }).round(2)
    
    print("Resumen por Estrategia:")
    display(strategy_summary)


## Visualizaciones


In [ ]:
if 'df_all' in locals():
    # Gráfico 1: Comparación de Tiempo Medio de Búsqueda
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_all, x='strategy', y='mean_search_time', ci='sd')
    plt.title('Comparación de Tiempo Medio de Búsqueda por Estrategia', fontsize=14, fontweight='bold')
    plt.xlabel('Estrategia', fontsize=12)
    plt.ylabel('Tiempo Medio (minutos)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
if 'df_all' in locals():
    # Gráfico 2: Comparación de Percentil 95
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_all, x='strategy', y='p95_search_time', ci='sd')
    plt.title('Comparación de Percentil 95 de Tiempo de Búsqueda', fontsize=14, fontweight='bold')
    plt.xlabel('Estrategia', fontsize=12)
    plt.ylabel('Tiempo P95 (minutos)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
if 'df_all' in locals():
    # Gráfico 3: Tasa de Error
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_all, x='strategy', y='error_rate', ci='sd')
    plt.title('Comparación de Tasa de Error por Estrategia', fontsize=14, fontweight='bold')
    plt.xlabel('Estrategia', fontsize=12)
    plt.ylabel('Tasa de Error', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
if 'df_all' in locals():
    # Gráfico 4: Utilización de Técnicos
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_all, x='strategy', y='avg_utilization', ci='sd')
    plt.title('Comparación de Utilización de Técnicos', fontsize=14, fontweight='bold')
    plt.xlabel('Estrategia', fontsize=12)
    plt.ylabel('Utilización Promedio (%)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Análisis de Escenarios Específicos


In [ ]:
if 'df_all' in locals():
    # Comparar escenarios de alta demanda
    high_demand = df_all[df_all['scenario'].str.contains('high_demand', na=False)]
    
    if not high_demand.empty:
        plt.figure(figsize=(12, 6))
        sns.barplot(data=high_demand, x='scenario', y='mean_search_time', hue='strategy')
        plt.title('Impacto de Alta Demanda en Tiempo de Búsqueda', fontsize=14, fontweight='bold')
        plt.xlabel('Escenario', fontsize=12)
        plt.ylabel('Tiempo Medio (minutos)', fontsize=12)
        plt.xticks(rotation=45)
        plt.legend(title='Estrategia')
        plt.tight_layout()
        plt.show()


In [ ]:
if 'df_all' in locals():
    # Comparar escenarios de mal ubicación
    misplaced = df_all[df_all['scenario'].str.contains('misplaced', na=False)]
    
    if not misplaced.empty:
        plt.figure(figsize=(12, 6))
        sns.barplot(data=misplaced, x='scenario', y='mean_search_time', hue='strategy')
        plt.title('Impacto de Nodos Mal Ubicados en Tiempo de Búsqueda', fontsize=14, fontweight='bold')
        plt.xlabel('Escenario', fontsize=12)
        plt.ylabel('Tiempo Medio (minutos)', fontsize=12)
        plt.xticks(rotation=45)
        plt.legend(title='Estrategia')
        plt.tight_layout()
        plt.show()


## Conclusiones


In [ ]:
if 'df_all' in locals():
    # Calcular reducción porcentual respecto al baseline
    baseline = df_all[df_all['strategy'] == 'actual']['mean_search_time'].mean()
    
    print("Reducción de Tiempo Medio respecto al Baseline:")
    print("=" * 50)
    
    for strategy in df_all['strategy'].unique():
        strategy_mean = df_all[df_all['strategy'] == strategy]['mean_search_time'].mean()
        reduction = ((baseline - strategy_mean) / baseline) * 100
        print(f"{strategy:15s}: {reduction:6.2f}%")
    
    print("\n" + "=" * 50)
    print(f"Baseline (actual): {baseline:.2f} minutos")
